# 3-4 業種の集まり方を読む・3-5 住まいと世帯から顧客像を描く / Reference notebook
`LANG` を選んで、すべてのセルを実行します。 / Choose `LANG` and run all cells.

In [ ]:
LANG = "ja"   # "ja" / "en"
BASE_URL = "https://raw.githubusercontent.com/YOUR_ACCOUNT/YOUR_REPO/main/data/"

In [ ]:
import os, subprocess, urllib.request
subprocess.run("pip install -q geopandas", shell=True)
for f in ['tokyo23_wards.geojson', 'stations_tokyo23_2023.csv', 'mesh500_industry_tokyo23.csv', 'mesh250_households_tokyo23.csv']:
    if not os.path.exists(f):
        try: urllib.request.urlretrieve(BASE_URL + f, f)
        except Exception as e: print(f, "をアップロードしてください / please upload", e)
if not os.path.exists("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"):
    subprocess.run("apt-get -y -qq install fonts-noto-cjk > /dev/null", shell=True)

In [ ]:
import numpy as np
def mesh_sw(code):
    c=str(code); lat=int(c[0:2])/1.5; lon=int(c[2:4])+100
    lat+=int(c[4])*5/60; lon+=int(c[5])*7.5/60
    lat+=int(c[6])*0.5/60; lon+=int(c[7])*0.75/60
    dlat,dlon=0.5/60,0.75/60
    for d in c[8:]:
        dlat/=2; dlon/=2; k=int(d)
        if k in (3,4): lat+=dlat
        if k in (2,4): lon+=dlon
    return lat,lon,dlat,dlon
def mesh_center(code):
    lat,lon,dlat,dlon=mesh_sw(code); return lat+dlat/2, lon+dlon/2
def mesh_polygon(code):
    from shapely.geometry import box
    lat,lon,dlat,dlon=mesh_sw(code); return box(lon,lat,lon+dlon,lat+dlat)


In [ ]:
import sys, os, re, json, glob, numpy as np, pandas as pd, geopandas as gpd, matplotlib
import matplotlib.pyplot as plt, matplotlib.patheffects as pe
from matplotlib import font_manager as fm
from matplotlib.patches import Patch
JP="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"; fm.fontManager.addfont(JP)
plt.rcParams.update({"font.family":fm.FontProperties(fname=JP).get_name(),"font.size":8,"savefig.dpi":300,"axes.spines.top":False,"axes.spines.right":False})
EN={"千代田区":"Chiyoda","中央区":"Chuo","港区":"Minato","新宿区":"Shinjuku","文京区":"Bunkyo","台東区":"Taito","墨田区":"Sumida","江東区":"Koto","品川区":"Shinagawa","目黒区":"Meguro","大田区":"Ota","世田谷区":"Setagaya","渋谷区":"Shibuya","中野区":"Nakano","杉並区":"Suginami","豊島区":"Toshima","北区":"Kita","荒川区":"Arakawa","板橋区":"Itabashi","練馬区":"Nerima","足立区":"Adachi","葛飾区":"Katsushika","江戸川区":"Edogawa"}
W=gpd.read_file("tokyo23_wards.geojson")
W6=W.to_crs(6677)
C5=["#f0f0f0","#c8c8c8","#969696","#636363","#252525"]
def read_estat(path):
    return pd.read_csv(path,encoding="shift_jis",skiprows=[1],dtype=str,low_memory=False)
def num(s): return pd.to_numeric(s.replace({"*":np.nan,"-":0}),errors="coerce")
def assign_ward(df,col="mesh_code"):
    cen=[mesh_center(str(c)) for c in df[col]]
    g=gpd.GeoDataFrame(df.copy(),geometry=gpd.points_from_xy([b for a,b in cen],[a for a,b in cen]),crs=4326)
    g["lat"]=[a for a,b in cen]; g["lon"]=[b for a,b in cen]
    j=gpd.sjoin(g,W[["ward_code","ward_ja","geometry"]],predicate="within")
    return pd.DataFrame(j.drop(columns=["geometry","index_right"]))
def meshgdf(df,col="mesh_code"):
    return gpd.GeoDataFrame(df.copy(),geometry=[mesh_polygon(str(c)) for c in df[col]],crs=4326).to_crs(6677)
def choro(ax,g,val,bins,labs,title,na_label=None,legend=True):
    g=g.copy(); g["cls"]=pd.cut(g[val],bins,labels=False,right=False)
    W6.plot(ax=ax,facecolor="white",edgecolor="none")
    gg=g[g.cls.notna()]; gg.plot(ax=ax,color=[C5[int(c)] for c in gg.cls],edgecolor="none")
    W6.boundary.plot(ax=ax,color="black",lw=0.35); ax.set_axis_off()
    if legend:
        h=[Patch(fc=c,ec="black",lw=0.4,label=l) for c,l in zip(C5,labs)]
        if na_label: h.append(Patch(fc="white",ec="black",lw=0.4,label=na_label))
        ax.legend(handles=h,title=title,loc="lower left",bbox_to_anchor=(1.0,0.0),fontsize=6.3,title_fontsize=7,frameon=False)
def gridlines(ax,im):
    nr,nc=im.get_array().shape
    ax.set_xticks(np.arange(-0.5,nc,1),minor=True); ax.set_yticks(np.arange(-0.5,nr,1),minor=True)
    ax.grid(which="minor",color="black",linewidth=0.8); ax.tick_params(which="minor",length=0)
    for s in ax.spines.values(): s.set_visible(False)


## 3-4 特化係数 / Location quotients

In [ ]:
J=LANG=="ja"; out=f"figures/{LANG}"; os.makedirs(out,exist_ok=True)
d=pd.read_csv("mesh500_industry_tokyo23.csv",dtype={"mesh_code":str})
INDS=["manufacturing","ict","wholesale_retail","finance","real_estate","professional","accommodation_food","medical_welfare"]
LJ={"manufacturing":"製造業","ict":"情報通信業","wholesale_retail":"卸売・小売業","finance":"金融・保険業","real_estate":"不動産業","professional":"専門・技術サービス業","accommodation_food":"宿泊・飲食サービス業","medical_welfare":"医療・福祉"}
LE={"manufacturing":"Manufacturing","ict":"ICT","wholesale_retail":"Wholesale/retail","finance":"Finance","real_estate":"Real estate","professional":"Professional services","accommodation_food":"Accommodation/food","medical_welfare":"Medical/welfare"}
nm=lambda k:(LJ if J else LE)[k]; wn=lambda w: w if J else EN[w]
tot=d[["emp_"+k for k in INDS]].sum()/d.emp_all.sum()
wd=d.groupby("ward_ja")[["emp_all"]+["emp_"+k for k in INDS]].sum()
lq=pd.DataFrame({k:(wd["emp_"+k]/wd.emp_all)/tot["emp_"+k] for k in INDS})
order=wd.emp_all.sort_values(ascending=False).index
lq=lq.loc[order]
fig,ax=plt.subplots(figsize=(4.5,4.3))
im=ax.imshow(lq.values,cmap="Greys",vmin=0,vmax=3,aspect="auto")
for i in range(lq.shape[0]):
    for j in range(lq.shape[1]):
        v=lq.values[i,j]; ax.text(j,i,f"{v:.1f}",ha="center",va="center",fontsize=5.8,color="white" if v>1.8 else "black")
ax.set_xticks(range(len(INDS)),[nm(k) for k in INDS],rotation=45,ha="right",rotation_mode="anchor",fontsize=6.5)
ax.set_yticks(range(len(lq)),[wn(w) for w in lq.index],fontsize=6.5); gridlines(ax,im)
cb=fig.colorbar(im,ax=ax,shrink=0.6); cb.ax.tick_params(labelsize=6.5)
fig.tight_layout(); fig.savefig(f"{out}/fig3-4-1_ward_lq.png",bbox_inches="tight"); plt.show()
# mesh LQ ict
m=d[d.emp_all>=1000].copy(); m["lq_ict"]=(m.emp_ict/m.emp_all)/tot["emp_ict"]
g=meshgdf(m); fig,ax=plt.subplots(figsize=(4.5,3.4))
choro(ax,g,"lq_ict",[0,0.5,1,2,3,99],["0.5未満","0.5〜1","1〜2","2〜3","3以上"] if J else ["< 0.5","0.5–1","1–2","2–3","≥ 3"],"情報通信業の特化係数" if J else "ICT location quotient","従業者1,000人未満" if J else "< 1,000 workers")
fig.savefig(f"{out}/fig3-4-3_mesh_lq_ict.png",bbox_inches="tight"); plt.show()
# count vs lq ward bar
cnt=wd.emp_ict.sort_values(ascending=False).head(10)
fig,axs=plt.subplots(1,2,figsize=(4.5,2.4),sharey=True)
yl=[wn(w) for w in cnt.index][::-1]
axs[0].barh(yl,cnt.values[::-1]/10000,color="#bbbbbb",edgecolor="black",lw=0.5)
axs[1].barh(yl,lq.loc[cnt.index,"ict"].values[::-1],color="#555555",edgecolor="black",lw=0.5)
axs[1].axvline(1,color="black",ls="--",lw=0.8)
for i,v in enumerate(cnt.values[::-1]/10000): axs[0].text(v+0.5,i,f"{v:.1f}",va="center",fontsize=6.5)
for i,v in enumerate(lq.loc[cnt.index,"ict"].values[::-1]): axs[1].text(v+0.05,i,f"{v:.2f}",va="center",fontsize=6.5)
axs[0].set_xlabel("情報通信業の従業者（万人）" if J else "ICT workers (10,000s)",fontsize=7); axs[1].set_xlabel("特化係数" if J else "Location quotient",fontsize=7)
axs[0].tick_params(labelsize=6.8); axs[1].tick_params(labelsize=6.8)
fig.tight_layout(); fig.savefig(f"{out}/fig3-4-2_ict_count_lq.png",bbox_inches="tight"); plt.show()
if J:
    st={"share23":tot.round(4).to_dict(),"lq_ward":lq.round(2).to_dict(),"ict_count":(wd.emp_ict).sort_values(ascending=False).head(10).astype(int).to_dict(),
        "n_mesh":len(d),"n_mesh1000":len(m),"lq_ict_ge3":int((m.lq_ict>=3).sum())}
    stall=pd.read_csv("stations_tokyo23_2023.csv")
    tops=m.sort_values("emp_ict",ascending=False).head(8)
    def near(lat,lon):
        dd=np.hypot((stall.lat-lat)*111,(stall.lon-lon)*91); i=dd.idxmin(); return stall.station_name[i]
    st["top_ict_mesh"]=[(r.ward_ja,near(r.lat,r.lon),int(r.emp_ict),int(r.emp_all),round(r.lq_ict,2)) for _,r in tops.iterrows()]
    toplq=m[m.emp_ict>=3000].sort_values("lq_ict",ascending=False).head(8)
    st["top_lq_mesh"]=[(r.ward_ja,near(r.lat,r.lon),int(r.emp_ict),int(r.emp_all),round(r.lq_ict,2)) for _,r in toplq.iterrows()]
    print(json.dumps(st,ensure_ascii=False)[:2000])
print("ok")


## 3-5 住まいと世帯 / Housing and households

In [ ]:
J=LANG=="ja"; out=f"figures/{LANG}"; os.makedirs(out,exist_ok=True)
d=pd.read_csv("mesh250_households_tokyo23.csv",dtype={"mesh_code":str})
d=d[d.hh_general>=100].copy()
d["r_single"]=d.hh_single/d.hh_general*100; d["r_under6"]=d.hh_with_under6/d.hh_general*100
d["r_rent"]=d.hh_private_rent/d.hh_in_housing*100; d["r_tower"]=d.hh_apt_15plus_bldg/d.hh_in_housing*100
g=meshgdf(d)
SPEC=[("r_single",[0,40,50,60,70,1e9],["40%未満","40〜50%","50〜60%","60〜70%","70%以上"],["< 40%","40–50%","50–60%","60–70%","≥ 70%"],"単身世帯の割合","One-person households"),
      ("r_under6",[0,4,6,8,10,1e9],["4%未満","4〜6%","6〜8%","8〜10%","10%以上"],["< 4%","4–6%","6–8%","8–10%","≥ 10%"],"6歳未満の子がいる世帯","Households with a child under 6"),
      ("r_rent",[0,30,40,50,60,1e9],["30%未満","30〜40%","40〜50%","50〜60%","60%以上"],["< 30%","30–40%","40–50%","50–60%","≥ 60%"],"民営の借家に住む世帯","Private rental households"),
      ("r_tower",[0,0.001,10,30,60,1e9],["0%","0〜10%","10〜30%","30〜60%","60%以上"],["0%","0–10%","10–30%","30–60%","≥ 60%"],"15階建以上の共同住宅","Buildings of 15+ floors")]
fig,axs=plt.subplots(2,2,figsize=(4.5,4.6))
for ax,(col,bins,lj,le,tj,te) in zip(axs.flat,SPEC):
    choro(ax,g,col,bins,None,None,legend=False)
    ax.set_title(tj if J else te,fontsize=7.5)
    h=[Patch(fc=c,ec="black",lw=0.4,label=l) for c,l in zip(C5,lj if J else le)]
    ax.legend(handles=h,loc="upper center",bbox_to_anchor=(0.5,0.02),ncol=3,fontsize=5.3,frameon=False,handlelength=1,columnspacing=0.6)
fig.tight_layout(h_pad=2.5); fig.savefig(f"{out}/fig3-5-1_households_maps.png",bbox_inches="tight"); plt.show()
w=d.groupby("ward_ja")[["hh_general","hh_single","hh_with_under6","hh_in_housing","hh_apt_15plus_bldg"]].sum()
w["single"]=w.hh_single/w.hh_general*100; w["u6"]=w.hh_with_under6/w.hh_general*100; w["tower"]=w.hh_apt_15plus_bldg/w.hh_in_housing*100
fig,ax=plt.subplots(figsize=(4.5,3.2))
ax.scatter(w.single,w.u6,s=[12+t*4 for t in w.tower],facecolor="white",edgecolor="black",lw=0.8)
halo=[pe.withStroke(linewidth=1.6,foreground="white")]
OFF={"品川区":(5,-9,"left"),"文京区":(5,3,"left"),"北区":(-5,-9,"right"),"墨田区":(5,-9,"left"),"目黒区":(5,4,"left"),"世田谷区":(4,-9,"left"),"荒川区":(0,8,"center"),"練馬区":(-4,-9,"right"),"大田区":(5,2,"left"),"板橋区":(5,-8,"left")}
for k,r in w.iterrows():
    dx,dy,ha=OFF.get(k,(4,3,"left")); ax.annotate(k if J else EN[k],(r.single,r.u6),xytext=(dx,dy),textcoords="offset points",fontsize=6.3,path_effects=halo,ha=ha)
ax.set_xlabel("単身世帯の割合（%）" if J else "One-person households (%)"); ax.set_ylabel("6歳未満の子がいる世帯の割合（%）" if J else "Households with a child under 6 (%)")
ax.text(0.01,0.02,"円の大きさ：15階建以上の共同住宅に住む世帯の割合" if J else "Circle size: share living in 15+ floor buildings",transform=ax.transAxes,ha="left",va="bottom",fontsize=6.3)
fig.tight_layout(); fig.savefig(f"{out}/fig3-5-2_ward_positioning.png",bbox_inches="tight"); plt.show()
print("ok")
